# Поиск гиперпараметров через Optuna

Ноутбук загружает признаки из `03a_data_and_features.ipynb` и выполняет поиск гиперпараметров через Optuna для LR, LightGBM, CatBoost, RF. Для Naive Bayes используется grid search по параметру `alpha`.

Результаты сохраняются в `data/interim/optuna_models.pkl`.

## Импорты и загрузка данных

In [12]:
import os
import sys
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from scipy.sparse import hstack

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.classical import (
    NUMERICAL_COLUMNS,
    prepare_features,
    train_classical_models,
    predict_message,
    save_models,
)
from src.evaluation.metrics import optimize_threshold
from src.config import PROCESSED_DIR, MODELS_DIR, INTERIM_DIR

import optuna
import lightgbm
import catboost
import nltk

nltk.download('stopwords', quiet=True)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings(
    'ignore',
    message='X does not have valid feature names',
    category=UserWarning,
    module='sklearn.utils.validation',
)
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

In [13]:
with open(INTERIM_DIR / 'features_train.pkl', 'rb') as f:
    train_data = pickle.load(f)
X_train = train_data['X_train']
X_train_nc = train_data['X_train_nc']
y_train = train_data['y_train']

print(f'X_train: {X_train.shape}')
print(f'X_train_nc: {X_train_nc.shape}')
print(f'y_train: {len(y_train)} меток')

X_train: (77523, 7020)
X_train_nc: (77523, 7020)
y_train: 77523 меток


## Настройка Optuna

Для каждой модели (кроме Naive Bayes) выполняется 30 trials с 3-fold Stratified CV. Оптимизация по F1-macro.

In [14]:
N_TRIALS = 15
CV_FOLDS = 3

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
optuna_results = {}

## Logistic Regression

Функция оптимизации гиперпараметров для Logistic Regression.

In [15]:
def objective_lr(trial):
    C = trial.suggest_float('C', 0.01, 10.0, log=True)

    model = LogisticRegression(
        C=C, solver='lbfgs',
        class_weight='balanced', max_iter=1000,
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

Запуск Optuna-оптимизации для Logistic Regression.

In [16]:
study_lr = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_lr.optimize(objective_lr, n_trials=N_TRIALS, show_progress_bar=True)

print(f'LR best F1-macro (CV): {study_lr.best_value:.4f}')
print(f'LR best params: {study_lr.best_params}')

  0%|          | 0/15 [00:00<?, ?it/s]

LR best F1-macro (CV): 0.9825
LR best params: {'C': 7.524538073376363}


Обучение лучшей версии Logistic Regression.

In [17]:
best_lr = LogisticRegression(
    **study_lr.best_params,
    solver='lbfgs',
    class_weight='balanced', max_iter=1000,
    random_state=RANDOM_STATE,
)
best_lr.fit(X_train, y_train)
optuna_results['lr'] = best_lr

## LightGBM

Функция оптимизации гиперпараметров для LightGBM.

In [18]:
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
    }
    model = lightgbm.LGBMClassifier(
        **params,
        class_weight='balanced',
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

Запуск Optuna-оптимизации для LightGBM.

In [19]:
study_lgbm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS, show_progress_bar=True)

print(f'LGBM best F1-macro (CV): {study_lgbm.best_value:.4f}')
print(f'LGBM best params: {study_lgbm.best_params}')

  0%|          | 0/15 [00:00<?, ?it/s]

LGBM best F1-macro (CV): 0.9820
LGBM best params: {'n_estimators': 198, 'max_depth': 6, 'learning_rate': 0.09941810361139941, 'num_leaves': 15, 'min_child_samples': 34, 'reg_alpha': 4.83276824082136, 'reg_lambda': 0.1430148311754847}


Обучение лучшей версии LightGBM.

In [20]:
best_lgbm = lightgbm.LGBMClassifier(
    **study_lgbm.best_params,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1,
)
best_lgbm.fit(X_train, y_train)
# LightGBM создаёт feature_names_in_ как property без возможности удаления,
# что вызывает варнинг sklearn при predict на разреженных матрицах.
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
    module="sklearn.utils.validation",
)
optuna_results['lgbm'] = best_lgbm

## CatBoost

Функция оптимизации гиперпараметров для CatBoost.

In [21]:
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 300),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
    }
    model = catboost.CatBoostClassifier(
        **params,
        auto_class_weights='Balanced',
        random_state=RANDOM_STATE,
        verbose=0,
    )
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=1)
    return scores.mean()

Запуск Optuna-оптимизации для CatBoost.

In [22]:
study_catboost = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_catboost.optimize(objective_catboost, n_trials=N_TRIALS, show_progress_bar=True)

print(f'CatBoost best F1-macro (CV): {study_catboost.best_value:.4f}')
print(f'CatBoost best params: {study_catboost.best_params}')

  0%|          | 0/15 [00:00<?, ?it/s]

CatBoost best F1-macro (CV): 0.9797
CatBoost best params: {'iterations': 284, 'depth': 7, 'learning_rate': 0.06644526695314788, 'l2_leaf_reg': 8.375955517619417}


Обучение лучшей версии CatBoost.

In [23]:
best_catboost = catboost.CatBoostClassifier(
    **study_catboost.best_params,
    auto_class_weights='Balanced',
    random_state=RANDOM_STATE,
    verbose=0,
)
best_catboost.fit(X_train, y_train)
optuna_results['catboost'] = best_catboost

## Random Forest

Функция оптимизации гиперпараметров для Random Forest.

In [24]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
    }
    model = RandomForestClassifier(
        **params,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

Запуск Optuna-оптимизации для Random Forest.

In [25]:
study_rf = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)

print(f'RF best F1-macro (CV): {study_rf.best_value:.4f}')
print(f'RF best params: {study_rf.best_params}')

  0%|          | 0/15 [00:00<?, ?it/s]

RF best F1-macro (CV): 0.9652
RF best params: {'n_estimators': 144, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6}


Обучение лучшей версии Random Forest.

In [26]:
best_rf = RandomForestClassifier(
    **study_rf.best_params,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
best_rf.fit(X_train, y_train)
optuna_results['rf'] = best_rf

## Naive Bayes (grid search по alpha)

Naive Bayes имеет один параметр `alpha` -- сглаживание Лапласа. Используется простой grid search вместо Optuna.

In [27]:
best_nb_alpha = 1.0
best_nb_f1 = 0.0
for alpha in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]:
    model = MultinomialNB(alpha=alpha)
    scores = cross_val_score(model, X_train_nc, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
    mean_f1 = scores.mean()
    if mean_f1 > best_nb_f1:
        best_nb_f1 = mean_f1
        best_nb_alpha = alpha

print(f'NB best alpha: {best_nb_alpha}, F1-macro (CV): {best_nb_f1:.4f}')

NB best alpha: 0.01, F1-macro (CV): 0.9051


Обучение лучшей версии Naive Bayes.

In [28]:
best_nb = MultinomialNB(alpha=best_nb_alpha)
best_nb.fit(X_train_nc, y_train)
optuna_results['nb'] = best_nb

## Сохранение моделей

Optuna-модели и study-объекты сохраняются в pickle для использования в последующих ноутбуках.

In [29]:
with open(INTERIM_DIR / 'optuna_models.pkl', 'wb') as f:
    pickle.dump({
        'models': optuna_results,
        'studies': {
            'lr': study_lr,
            'lgbm': study_lgbm,
            'catboost': study_catboost,
            'rf': study_rf,
        },
    }, f)

print(f'Сохранено: {INTERIM_DIR / "optuna_models.pkl"}')
print(f'Моделей: {len(optuna_results)}')
print(f'Модели: {list(optuna_results.keys())}')

Сохранено: /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/optuna_models.pkl
Моделей: 5
Модели: ['lr', 'lgbm', 'catboost', 'rf', 'nb']
